# Companion notebook (ខ្មែរ) — `ch05/15_tiny-aya/standalone-tiny-aya-plus-kv-cache.ipynb`

> **ប្រភព និងអាជ្ញាបណ្ណ** — កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង
> [`ch05/15_tiny-aya/standalone-tiny-aya-plus-kv-cache.ipynb`](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/15_tiny-aya/standalone-tiny-aya-plus-kv-cache.ipynb)
> នៃឃ្លាំង [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch)
> (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។ Comments ក្នុងកូដត្រូវបានបកប្រែជាខ្មែរដោយសហគមន៍។
> Notebook នេះ **មិនមានអត្ថបទសៀវភៅទេ** (អត្ថបទសៀវភៅជាកម្មសិទ្ធិ Manning ដកចេញពីអាជ្ញាបណ្ណបើកចំហ) —
> សម្រាប់ការពន្យល់សូមមើលសៀវភៅផ្លូវការ ឬ notebook ដើម។

In [ ]:
# ការតភ្ជាប់ឯកសារជំនួយ — ឱ្យដំណើរការភ្ជាប់ដូច notebook ដើមរបស់ upstream
# (ចង្អុល sys.path និងថតធ្វើការទៅកាន់ថតកូដក្នុងឃ្លាំងកសារខ្មែរនេះ)
import os, sys
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "កូដ")):
    _p = os.path.dirname(_p)
_UP = os.path.join(_p, "កូដ", "ch05/15_tiny-aya")
sys.path.insert(0, _UP)
os.chdir(_UP)


In [ ]:
# pip install -r https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch05/07_gpt_to_llama/requirements-extra.txt

In [ ]:
from importlib.metadata import version

pkgs = [
    #"blobfile",         # to download pretrained weights
    "huggingface_hub",  # to download pretrained weights
    "tiktoken",         # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
from pathlib import Path

REPO_ID = "CohereLabs/tiny-aya-global"
#REPO_ID = "CohereLabs/tiny-aya-fire" 
#REPO_ID = "CohereLabs/tiny-aya-water"
#REPO_ID = "CohereLabs/tiny-aya-earth"

LOCAL_DIR = Path(REPO_ID).parts[-1]

In [ ]:
import torch
import torch.nn as nn


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)

In [ ]:
# Aya uses a bias-less LayerNorm variant. 
# The difference to classic LayerNorm is that it only 
# has a scale parameter (weight), no shift parameter (bias).

class CohereLayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb_dim))

    def forward(self, x):
        input_dtype = x.dtype
        x = x.to(torch.float32)
        mean = x.mean(dim=-1, keepdim=True)
        variance = (x - mean).pow(2).mean(dim=-1, keepdim=True)
        x = (x - mean) * torch.rsqrt(variance + self.eps)
        return (self.weight.to(torch.float32) * x).to(input_dtype)

In [ ]:
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "head_dim must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (
        theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim)
    )
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Cohere uses interleaved even/odd angle layout per head-dim pair.
    # Llama2 notebook examples often use a split-halves layout via cat([angles, angles]).
    # Both are equivalent only when paired with the matching rotate logic:
    # - interleaved layout -> even/odd rotation implementation (below)
    # - split-halves layout -> half/half rotate implementation
    angles = torch.repeat_interleave(angles, 2, dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin, offset=0):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "head_dim must be even"

    # Split x into even and odd components (interleaved layout)
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]

    # Adjust sin and cos shapes
    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    x_float = x.float()
    rotated = torch.stack((-x_odd.float(), x_even.float()), dim=-1).flatten(-2)
    x_rotated = (x_float * cos) + (rotated * sin)

    return x_rotated.to(dtype=x.dtype)

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(
        self,
        d_in,
        num_heads,
        num_kv_groups,
        head_dim=None,
        qk_norm=False,
        attention_bias=False,
        dtype=None,
        attn_type="full_attention",
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim
        self.attn_type = attn_type

        self.W_query = nn.Linear(
            d_in,
            self.d_out,
            bias=attention_bias,
            dtype=dtype,
        )
        self.W_key = nn.Linear(
            d_in,
            num_kv_groups * head_dim,
            bias=attention_bias,
            dtype=dtype,
        )
        self.W_value = nn.Linear(
            d_in,
            num_kv_groups * head_dim,
            bias=attention_bias,
            dtype=dtype,
        )
        self.out_proj = nn.Linear(
            self.d_out,
            d_in,
            bias=attention_bias,
            dtype=dtype,
        )

        if qk_norm:
            self.q_norm = CohereLayerNorm(head_dim, eps=1e-6)
            self.k_norm = CohereLayerNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        b, num_tokens, _ = x.shape

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Reshape
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys_new = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values_new = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # Optional normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys_new = self.k_norm(keys_new)

        # Cohere2 applies RoPE only on sliding-attention layers.
        if self.attn_type == "sliding_attention":
            queries = apply_rope(queries, cos, sin, offset=start_pos)
            keys_new = apply_rope(keys_new, cos, sin, offset=start_pos)

        if cache is not None:
            prev_k, prev_v = cache
            keys = torch.cat([prev_k, keys_new], dim=2)
            values = torch.cat([prev_v, values_new], dim=2)
            next_cache = (keys, values)
        else:
            keys, values = keys_new, values_new
            next_cache = (keys, values)

        # Expand K and V to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # Attention
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1, dtype=torch.float32).to(queries.dtype)
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)

        return self.out_proj(context), next_cache

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg, attn_type):
        super().__init__()
        self.attn_type = attn_type

        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_heads"],
            head_dim=cfg["head_dim"],
            qk_norm=False,
            attention_bias=cfg["attention_bias"],
            dtype=cfg["dtype"],
            attn_type=attn_type,
        )
        self.ff = FeedForward(cfg)
        self.input_layernorm = CohereLayerNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])

    def forward(self, x, mask_global, mask_local, cos, sin, start_pos=0, cache=None):
        attn_mask = mask_local if self.attn_type == "sliding_attention" else mask_global

        shortcut = x
        x = self.input_layernorm(x)
        x_attn, next_cache = self.att(
            x,
            attn_mask,
            cos,
            sin,
            start_pos=start_pos,
            cache=cache,
        )  # Shape [batch_size, num_tokens, emb_dim]
        x_ff = self.ff(x)

        # Cohere2 parallel residual block
        x = shortcut + x_attn + x_ff
        return x, next_cache

In [ ]:
class TinyAyaModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert len(cfg["layer_types"]) == cfg["n_layers"], "layer_types must match n_layers"

        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        self.trf_blocks = nn.ModuleList([TransformerBlock(cfg, t) for t in cfg["layer_types"]])

        self.final_norm = CohereLayerNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        self.logit_scale = cfg["logit_scale"]

        cos, sin = compute_rope_params(
            head_dim=cfg["head_dim"],
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        if cfg["tie_word_embeddings"]:
            self.out_head.weight = self.tok_emb.weight

        self.current_pos = 0  # Track current position in KV cache

    def create_masks(self, num_tokens, device, pos_start=0, total_kv_tokens=None):
        if total_kv_tokens is None:
            total_kv_tokens = pos_start + num_tokens

        query_positions = torch.arange(pos_start, pos_start + num_tokens, device=device).unsqueeze(1)
        key_positions = torch.arange(total_kv_tokens, device=device).unsqueeze(0)

        # Future mask
        mask_global = key_positions > query_positions

        # Sliding-window mask
        far_past = key_positions + self.cfg["sliding_window"] <= query_positions
        mask_local = mask_global | far_past

        # Expand to [batch, heads, seq, seq]-broadcastable shape
        return mask_global.unsqueeze(0).unsqueeze(0), mask_local.unsqueeze(0).unsqueeze(0)

    def forward(self, input_ids, attention_mask=None, cache=None):
        tok_embeds = self.tok_emb(input_ids)
        x = tok_embeds
        num_tokens = x.shape[1]

        if cache is not None:
            pos_start = self.current_pos
            pos_end = pos_start + num_tokens
            self.current_pos = pos_end
            total_kv_tokens = pos_end
        else:
            pos_start = 0
            total_kv_tokens = num_tokens

        mask_global, mask_local = self.create_masks(
            num_tokens,
            x.device,
            pos_start=pos_start,
            total_kv_tokens=total_kv_tokens,
        )

        if attention_mask is not None:
            # True means mask in this implementation.
            pad_mask = attention_mask[:, None, None, :total_kv_tokens].to(dtype=torch.bool).logical_not()
            mask_global = mask_global | pad_mask
            mask_local = mask_local | pad_mask

        cos = self.cos.to(x.device, dtype=x.dtype)
        sin = self.sin.to(x.device, dtype=x.dtype)

        for i, block in enumerate(self.trf_blocks):
            blk_cache = cache.get(i) if cache else None
            x, new_blk_cache = block(
                x,
                mask_global,
                mask_local,
                cos,
                sin,
                start_pos=pos_start,
                cache=blk_cache,
            )
            if cache is not None:
                cache.update(i, new_blk_cache)

        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits * self.logit_scale

    def reset_kv_cache(self):
        self.current_pos = 0


class KVCache:
    def __init__(self, n_layers):
        self.cache = [None] * n_layers

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    def reset(self):
        for i in range(len(self.cache)):
            self.cache[i] = None

In [ ]:
TINY_AYA_CONFIG = {
    "vocab_size": 262_144,  # ទំហំ vocabulary
    "context_length": 500_000,        # Context length in the HF config
    "emb_dim": 2048,  # វិមាត្រ embedding
    "n_heads": 16,  # ចំនួន attention heads
    "n_layers": 36,  # ចំនួនស្រទាប់
    "hidden_dim": 11_008,             # Size of the intermediate dimension in FeedForward
    "head_dim": 128,                  # Size of the heads in GQA
    "n_kv_heads": 4,                  # Number of KV heads for grouped-query attention
    "attention_bias": False,          # Whether attention projections use bias terms
    "attention_dropout": 0.0,         # Attention dropout
    "sliding_window": 4096,           # Sliding-window attention context
    "layer_types": [
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
    ],
    "rope_base": 50_000.0,            # The base in RoPE's "theta"
    "layer_norm_eps": 1e-5,           # Epsilon used by layer normalization
    "logit_scale": 1.0,               # Final logits scaling factor
    "tie_word_embeddings": True,      # Whether input embedding and output head are tied
    "bos_token_id": 2,
    "eos_token_id": 3,
    "pad_token_id": 0,
    "dtype": torch.bfloat16,          # Lower-precision dtype to reduce memory usage
}

In [ ]:
model = TinyAyaModel(TINY_AYA_CONFIG)

In [ ]:
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

In [ ]:
from tokenizers import Tokenizer


class TinyAyaTokenizer:
    def __init__(self, tokenizer_file_path, eos_token_id=3, pad_token_id=0, bos_token_id=2):
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))

        eos_from_tok = self._tok.token_to_id("<EOS_TOKEN>")
        pad_from_tok = self._tok.token_to_id("<PAD>")
        bos_from_tok = self._tok.token_to_id("<BOS_TOKEN>")

        self.eos_token_id = eos_from_tok if eos_from_tok is not None else eos_token_id
        self.pad_token_id = pad_from_tok if pad_from_tok is not None else pad_token_id
        self.bos_token_id = bos_from_tok if bos_from_tok is not None else bos_token_id

    def encode(self, text):
        return self._tok.encode(text).ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)


def apply_chat_template(user_text):
    return (
        "<BOS_TOKEN>"
        "<|START_OF_TURN_TOKEN|><|USER_TOKEN|>"
        f"{user_text}"
        "<|END_OF_TURN_TOKEN|>"
        "<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>"
    )

In [ ]:
print(REPO_ID)

In [ ]:
# Uncomment and run the following code if you are executing the notebook for the first time

from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import hf_hub_download

tokenizer_file_path = Path(LOCAL_DIR) / "tokenizer.json"
if not tokenizer_file_path.exists():
    try:
        tokenizer_file_path = hf_hub_download(repo_id=REPO_ID, filename="tokenizer.json", local_dir=LOCAL_DIR)
    except Exception as e:
        print(f"Warning: failed to download tokenizer.json: {e}")
        tokenizer_file_path = "tokenizer.json"

In [ ]:
tokenizer = TinyAyaTokenizer(
    tokenizer_file_path=Path(LOCAL_DIR) / "tokenizer.json",
    eos_token_id=TINY_AYA_CONFIG["eos_token_id"],
    pad_token_id=TINY_AYA_CONFIG["pad_token_id"],
    bos_token_id=TINY_AYA_CONFIG["bos_token_id"],
)

prompt = apply_chat_template("Give me a short introduction to large language models in 3 sentences.")
input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

In [ ]:
def load_weights_into_tiny_aya(model, param_config, params):
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}"
            )

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right.to(dtype=left.dtype, device=left.device))
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    model.tok_emb.weight = assign(
        model.tok_emb.weight,
        params["model.embed_tokens.weight"],
        "model.embed_tokens.weight",
    )

    for l in range(param_config["n_layers"]):
        block = model.trf_blocks[l]
        att = block.att

        # Q, K, V projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight",
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight",
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight",
        )

        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight",
        )

        # Feedforward weights
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight",
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight",
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight",
        )

        # Layernorm
        block.input_layernorm.weight = assign(
            block.input_layernorm.weight,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight",
        )

    # Final normalization and output head
    model.final_norm.weight = assign(
        model.final_norm.weight,
        params["model.norm.weight"],
        "model.norm.weight",
    )

    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        if param_config["tie_word_embeddings"]:
            model.out_head.weight = model.tok_emb.weight
            print("Model uses weight tying.")

In [ ]:
import json
from safetensors.torch import load_file
from huggingface_hub import snapshot_download


repo_dir = snapshot_download(repo_id=REPO_ID, local_dir=LOCAL_DIR)
index_path = Path(repo_dir) / "model.safetensors.index.json"
with open(index_path, "r") as f:
    index = json.load(f)

weights_dict = {}
for filename in sorted(set(index["weight_map"].values())):
    shard_path = Path(repo_dir) / filename
    shard = load_file(shard_path)
    weights_dict.update(shard)

load_weights_into_tiny_aya(model, TINY_AYA_CONFIG, weights_dict)
model.to(device)
del weights_dict

In [ ]:
def count_unique_parameters(model):
    unique_params = set()
    total_unique_params = 0
    
    for param in model.parameters():
        if param.data_ptr() not in unique_params:
            total_unique_params += param.numel()
            unique_params.add(param.data_ptr())
            
    return total_unique_params

total_params_uniq = count_unique_parameters(model)
print(f"Total number of unique parameters: {total_params_uniq:,}")

In [ ]:
import warnings


stop_ids = {
    tokenizer.eos_token_id,
    tokenizer._tok.token_to_id("<|END_RESPONSE|>"),
    tokenizer._tok.token_to_id("<|END_OF_TURN_TOKEN|>"),
}
stop_ids = {x for x in stop_ids if x is not None}


def generate_text_basic_stream(
    model,
    token_ids,
    max_new_tokens,
    stop_token_ids=None,
    context_size=None,
):
    # context_size is accepted for backward compatibility and has no effect.
    if context_size is not None:
        warnings.warn(
            "context_size is ignored by this streaming generator. Prompts are not "
            "truncated; the model's configured context limit still applies.",
            stacklevel=2,
        )

    stop_token_ids = set(stop_token_ids or [])

    model.eval()
    with torch.no_grad():
        cache = KVCache(n_layers=model.cfg["n_layers"])
        model.reset_kv_cache()

        # Prime the cache with the initial context
        logits = model(token_ids, cache=cache)

        for step in range(max_new_tokens):
            next_token = torch.argmax(logits[:, -1], dim=-1, keepdim=True)

            if stop_token_ids and next_token.item() in stop_token_ids:
                break

            yield next_token

            token_ids = torch.cat([token_ids, next_token], dim=1)
            # Feed only the new token to the model; cache handles history
            if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
                logits = model(next_token, cache=cache)

In [ ]:
prompt = apply_chat_template("Give me a short introduction to large language models in 3 sentences.")
input_token_ids = tokenizer.encode(prompt)
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)


if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    stop_token_ids=stop_ids
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"
    
    print(f"\n\nGPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ